# K6: TF-IDF, logistička regresija naspram stabla odlučivanja (online prodavnica odeće)
**Tip:** Blok 2, zadatak 2. Reprezentacija: TF-IDF. Dva klasifikatora koja se porede, pa podešavanje stabla.

### Zadatak
U fajlu `E-Commerce_Reviews_for_class.csv` su komentari kupaca online prodavnice odeće (`Clothing ID`, `Age`, `Title`, `Review Text`, `Rating`).
- Napraviti kolonu `review_content` spajanjem `Title` i `Review Text`.
- Od kolone `Rating` napraviti `sentiment` (1 i 2 negativan, 3 neutralan, 4 i 5 pozitivan).
- Transformisati `review_content` primenom **TF-IDF**. Proceniti da li još neku varijablu treba uključiti.
- Prvi model: **logistička regresija**, podrazumevani hiperparametri. Drugi model: **stablo odlučivanja**, podrazumevani hiperparametri. Uporediti ih.
- Treći model: **podešeno stablo** (jedan hiperparametar za TF-IDF, jedan za stablo, 5-fold kros-validacija), uz komentar šta bi sve podešavao.
- Uporediti sva tri modela i `DummyClassifier` kao donju granicu.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci, napravljena `review_content` (prazan naslov se menja praznim stringom da ne "pokvari" spoj).
- Napravljena izlazna varijabla `sentiment` (pozitivnih je najviše).
- Pregledani `Age` (kdeplot) i `Clothing ID` (broj vrednosti).
- Odluka: ulaz je samo `review_content`.
- Podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'E-Commerce_Reviews_for_class.csv')
data.info()

In [ ]:
data['Title'] = data['Title'].fillna('')
data = data.dropna(subset=['Review Text']).reset_index(drop=True)
data['review_content'] = data['Title'] + '. ' + data['Review Text']
print(data['review_content'].head(3).tolist())

In [ ]:
def rating_to_sentiment(rating: int):
    if rating < 3:
        return 'negativan'
    return 'neutralan' if rating == 3 else 'pozitivan'

data['sentiment'] = data['Rating'].apply(rating_to_sentiment)
print(data['sentiment'].value_counts(normalize=True))

In [ ]:
plt.figure(figsize=(10, 6))
sb.kdeplot(data=data, x='Age', hue='sentiment',
           fill=True, common_norm=False, alpha=0.5, linewidth=2)
plt.title('Starost kupca po sentimentu')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print('Broj razlicitih proizvoda:', data['Clothing ID'].nunique())

In [ ]:
X = data['review_content']
y = data['sentiment']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

In [ ]:
#logisticka , stablo uporediti , gridseaerch , dummy

In [ ]:
nlp_pipe=spacy.load('en_core_web_sm')

In [ ]:
def spacy_sredi_tekst(doc):
    tokens=nlp_pipe(doc)
    ocisceniTokeni=[]
    for token in tokens:
        lemma_lower=token.lemma_.lower()
        if lemma_lower in ['not','no']:
            ocisceniTokeni.append(lemma_lower)
            continue
        if token.is_stop or token.is_punct or token.like_url or token.is_space or token.like_email or token.like_url:
            continue
        ocisceniTokeni.append(lemma_lower)
    return ocisceniTokeni


In [ ]:
vectorizer=TfidfVectorizer(
        tokenizer=spacy_sredi_tekst,
        analyzer="word",
        stop_words=None,
        token_pattern=None,
        ngram_range=(1, 2),
        max_df=0.95,
        min_df=0.01,
        max_features=None)

In [ ]:
logisticka=LogisticRegression(class_weight='balanced',max_iter=1500,C=1,random_state=42)

In [ ]:
stablo=DecisionTreeClassifier(random_state=42,class_weight='balanced')

In [ ]:
# #kako imamo mnogo ne balansiran data set koristimo smote
# pozitivan    0.775667
# neutralan    0.115000
# negativan    0.109333

In [ ]:
pipe_log=Pipeline([
    ('vectorizer',vectorizer),
    ('smote',SMOTE(random_state=42)),
    ('cls_log',logisticka)
])

In [ ]:
pipe_stablo=Pipeline([
    ('vectorizer',vectorizer),
    ('smote',SMOTE(random_state=42)),
    ('cls_stablo',stablo)
])

In [ ]:
model_log=pipe_log.fit(X_train,y_train)

In [ ]:
model_stablo=pipe_stablo.fit(X_train,y_train)

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)

In [ ]:
predict_dummy=dummy_clf.predict(X_test)

In [ ]:
predict_stablo=pipe_stablo.predict(X_test)

In [ ]:
predict_log=pipe_log.predict(X_test)

In [ ]:
params={
    'cls_stablo__max_depth':[10,25,50],
    'vectorizer__min_df':[0.05,1,3]
}

In [ ]:
cv=GridSearchCV(pipe_stablo,param_grid=params,scoring='f1_macro',cv=5,verbose=2)

In [ ]:
cv=cv.fit(X_train,y_train)

In [ ]:
def get_model_metrics(y_true:np.ndarray, y_pred:np.ndarray, model_name:str):

    # compute P, R, F1 for both averages
    report = classification_report(y_true, y_pred, output_dict=True)

    # compute balanced accuracy
    b_acc = balanced_accuracy_score(y_true, y_pred)

    # extract metrics for the specified averages
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']

    results = {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': b_acc
    }

    return results

In [ ]:
cv_model=cv.best_estimator_

In [ ]:
predict_cv=cv_model.predict(X_test)

In [ ]:
pd.DataFrame([get_model_metrics(y_test,predict_dummy,"DUMMY"),get_model_metrics(y_test,predict_log,"LOG MODEL"),get_model_metrics(y_test,predict_stablo,"STABLO 1"),get_model_metrics(y_test,predict_cv,"STABLO IZ CV")]).round(2)